# TwoR2C_TiTe — VUBC

2R2C (states $T_i$, $T_e$), temperature task: $T_i$ supervised.

Prepares `data_train.joblib` and `identification_config.json`; the model is trained with `rct-train` (last cell). Data split as in `six_r3c_eniso13790/VUBC`: season w2324, training windows of 3 h sliding by 1 h from 1 Dec 2023 to 1 Feb 2024, 24 h pre-window lag, no validation split.

In [1]:
import json
import os
from datetime import datetime, timedelta

import joblib
import pandas as pd

from rctoolbox.utils.project_root import find_project_root
from rctoolbox.data.time_windows_generator import generate_time_windows
from rctoolbox.data.numpydataloader import DataLoader as NumpyDataLoader

project_root = find_project_root()
os.chdir(project_root)

In [2]:
HEMS_ID = "VUBC"
OUT_DIR_PATH = project_root / "examples" / "run_identification" / "TwoR2C_TiTe" / HEMS_ID

## DATA

In [3]:
DATA_DIR = project_root / "examples" / "data"
DATA = pd.read_pickle(DATA_DIR / f"{HEMS_ID}_data.pkl")
SEASON = "w2324"
df = DATA[SEASON]["df"]

print("season:        ", SEASON)
print("df shape:      ", df.shape)
print("time range:    ", df.index.min(), "->", df.index.max())

season:         w2324
df shape:       (3721, 10)
time range:     2023-10-30 00:00:00 -> 2024-04-02 00:00:00


### TRAIN SPLIT

In [4]:
wdh = 3; spb = int(1e10); ssh = 1
pre_window_lag = timedelta(hours=24)

t1_train = datetime(2023, 12, 1)
t2_train = datetime(2024, 2, 1)
tws_train = generate_time_windows(start_time=t1_train, end_time=t2_train, window_duration_hours=wdh, samples_per_batch=spb, slide_step_hours=ssh)

tws = {"train": tws_train}
dataloader = NumpyDataLoader(df)
dataloader.load_all_data_batches(time_windows=tws, features=df.columns.tolist(), pre_window_lag=pre_window_lag)

In [5]:
from rctoolbox.models.rc.TwoR2C_TiTe.build_data_dict import build_data_dict
COLUMNS = None   # model signal -> data column; defaults: T_i = y1, T_a, Phi_sol, Phi_HC
build = lambda split: build_data_dict(dataloader, split, columns=COLUMNS)

data_joblib = {"train": build("train"), "val": None, "columns": COLUMNS}
data_joblib_path = OUT_DIR_PATH / "data_train.joblib"
joblib.dump(data_joblib, data_joblib_path)
print("train windows:", data_joblib["train"]["t"].shape)

train windows: (4,)


## PARAMETER BOUNDS

The bounds are derived from the building geometry (the VUBC building config); `FREE_PARAM_BOUNDS` replaces those of the parameters listed and goes into `model_config["free_param_bounds"]`.

In [6]:
from rctoolbox.utils.config_loader import load_default_building_config
from rctoolbox.models.rc.TwoR2C_TiTe.parameter_space import ParameterSpace_2R2C

ps_default = ParameterSpace_2R2C(load_default_building_config(HEMS_ID))

# Bounds hit by the calibration, widened on that side: R_ea (lower, 10x) and C_i (upper) in the first
# run; C_i again (to 1e9 J/K, the C_i upper bound of the 3R3C/6R3C) and k_sol (upper, ~3x) in the second.
# The other parameters keep the bounds derived from the building geometry.
FREE_PARAM_BOUNDS = {
    "R_ea":  [3.0e-5, ps_default.free_param_bounds["R_ea"][1]],   # default lower 3.0e-4 K/W
    "C_i":   [ps_default.free_param_bounds["C_i"][0], 1.0e9],     # default upper 2.13e7 J/K
    "k_sol": [ps_default.free_param_bounds["k_sol"][0], 450.0],   # default upper 144 m2
}
pd.DataFrame({"default": ps_default.free_param_bounds, "used": {**ps_default.free_param_bounds, **{k: tuple(v) for k, v in FREE_PARAM_BOUNDS.items()}}})

,default,used
R_ie,"(8.056005349187552e-05, 0.00648508430609598)","(8.056005349187552e-05, 0.00648508430609598)"
R_ea,"(0.00030047992774433403, 0.01774500637111553)","(3e-05, 0.01774500637111553)"
C_i,"(407088.0, 21279599.999999996)","(407088.0, 1000000000.0)"
C_e,"(1542000.0, 177329999.99999997)","(1542000.0, 177329999.99999997)"
k_sol,"(2.4587103936820216, 144.0680042549351)","(2.4587103936820216, 450.0)"


## IDENTIFICATION CONFIG

In [7]:
model_config = {
    "model_type":  "2R2C_temperature",
    "init_config": {"method": "dynamic", "lag_h": 24},   # lag window used [h], <= pre_window_lag
}
metric = "RMSE"
optimization_config = {"objective_function": metric, "std_weight": 0.0, "pop_size": 100, "n_gen": 100}

model_config["free_param_bounds"] = FREE_PARAM_BOUNDS
selection_config  = {"criteria": "train error", "select_R_star": True, "threshold_config": {"type": "percentage", "x": 0.05}}
clustering_config = {"min_samples_ratio": 0.1, "cl_eval_metric": metric, "cl_eval_criteria": "train error"}

identification_config = {
    "model_config":        model_config,
    "optimization_config": optimization_config,
    "selection_config":    selection_config,
    "clustering_config":   clustering_config,
}
with open(OUT_DIR_PATH / "identification_config.json", "w") as f:
    json.dump(identification_config, f, indent=4)

## TRAIN (CLI)

`--building-config VUBC`: the bounds are derived from the building geometry.